# God-Shot Evaluation Report & Charts (S11)

Owner: **Rishika** (charts & report) · **Siva** (`eval/score.py`, `eval/run_all.py`).

Generates:
- `eval/figs/accuracy_by_type.png` (grouped baseline vs loop bars with 95% clip-bootstrap CIs)
- `eval/figs/accuracy_by_group.png` (CameraBench motion-only vs own clips layout+motion, plus difficulty breakdown)
- `eval/figs/calibration.png` (5-bin reliability diagram, ECE, Brier score)
- `eval/figs/selective_accuracy.png` (high-confidence vs `low_confidence` accuracy)
- `eval/figs/loop_behavior.png` (rounds-to-converge histogram, regressions, % hurt, latency)
- `eval/results.md` (single source of truth for all slide numbers — includes split & freeze time)
- `eval/failures.md` (every failed label with frame thumbnail + scene claim for failure gallery selection)

In [ ]:
from __future__ import annotations

import json
import math
import sys
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "godshot").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from godshot.schema import LabelV1, SceneV1, load_checks, load_labels  # noqa: E402

# --- Run configuration ---
SPLIT = "dev"  # Switch to "test" ONLY after prompt/threshold freeze
FREEZE_TIME = "TBD (update at freeze before running test)"
BOOTSTRAP_N = 1000
BOOTSTRAP_SEED = 0

LABELS_PATH = REPO_ROOT / "eval" / "labels" / "labels.jsonl"
RUNS_DIR = REPO_ROOT / "eval" / "runs"
FIGS_DIR = REPO_ROOT / "eval" / "figs"
RESULTS_MD = REPO_ROOT / "eval" / "results.md"
FAILURES_MD = REPO_ROOT / "eval" / "failures.md"
FIGS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "font.size": 11,
    }
)
print(f"Repo: {REPO_ROOT} | Split: {SPLIT} | Labels exist: {LABELS_PATH.exists()}")

## 1. Statistical Helpers (Clip Bootstrap CI, Paired Sign Test, ECE & Brier)

In [ ]:
def bootstrap_ci_by_clip(
    df: pd.DataFrame,
    col: str = "correct",
    n_boot: int = BOOTSTRAP_N,
    seed: int = BOOTSTRAP_SEED,
) -> tuple[float, float, float]:
    """Mean accuracy and 95% CI resampling clips with replacement (cluster bootstrap)."""
    if df.empty:
        return float("nan"), float("nan"), float("nan")
    clips = df["clip_id"].unique()
    grouped = {cid: df.loc[df["clip_id"] == cid, col].to_numpy(dtype=float) for cid in clips}
    mean = float(df[col].mean())
    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        sampled = rng.choice(clips, size=len(clips), replace=True)
        vals = np.concatenate([grouped[c] for c in sampled])
        boots[i] = float(vals.mean())
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return mean, float(lo), float(hi)


def paired_sign_test(base_by_clip: pd.Series, full_by_clip: pd.Series) -> dict[str, float | int]:
    """Two-sided exact binomial sign test over per-clip accuracy differences (full - baseline)."""
    common = base_by_clip.index.intersection(full_by_clip.index)
    diff = full_by_clip.loc[common] - base_by_clip.loc[common]
    wins = int((diff > 1e-9).sum())
    losses = int((diff < -1e-9).sum())
    ties = len(common) - wins - losses
    n = wins + losses
    if n == 0:
        p_val = 1.0
    else:
        k = min(wins, losses)
        tail = sum(math.comb(n, i) for i in range(k + 1)) / (2**n)
        p_val = min(1.0, 2.0 * tail)
    return {"n_clips": len(common), "wins": wins, "losses": losses, "ties": ties, "p_value": p_val}


def calibration_stats(probs: np.ndarray, truths: np.ndarray, n_bins: int = 5) -> dict:
    """5-bin reliability curve, ECE, and Brier score for noul predictions vs binary human truth."""
    if len(probs) == 0:
        return {"ece": float("nan"), "brier": float("nan"), "bins": []}
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    brier = float(np.mean((probs - truths) ** 2))
    ece = 0.0
    bins = []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (probs >= lo) & (probs <= hi if i == n_bins - 1 else probs < hi)
        cnt = int(mask.sum())
        if cnt > 0:
            conf = float(probs[mask].mean())
            acc = float(truths[mask].mean())
            ece += (cnt / len(probs)) * abs(acc - conf)
            bins.append({"bin": f"{lo:.1f}-{hi:.1f}", "count": cnt, "conf": conf, "acc": acc})
    return {"ece": ece, "brier": brier, "bins": bins}

## 2. Load Labels, Runs, and Score Against Human Truth

In [ ]:
from eval.score import score_scene

all_labels: list[LabelV1] = load_labels(LABELS_PATH) if LABELS_PATH.exists() else []
# Keep primary labeler per clip for headline scoring; retain all records for IAA.
split_labels = [lb for lb in all_labels if lb.split == SPLIT]
primary_by_clip: dict[str, LabelV1] = {}
for lb in split_labels:
    primary_by_clip.setdefault(lb.clip_id, lb)

rows: list[dict] = []
loop_rows: list[dict] = []
control_p_yes: list[float] = []

for clip_id, label_rec in primary_by_clip.items():
    group = "camerabench" if label_rec.source == "camerabench" else "own_clips"
    for system, fname in (("baseline", "scene_r0.json"), ("full", "scene_final.json")):
        run_dir = RUNS_DIR / system / clip_id
        scene_path = run_dir / fname
        if not scene_path.exists():
            continue
        scene = SceneV1.load(scene_path)
        low_conf_entities = {lc.entity_id for lc in scene.low_confidence}
        scored = score_scene(scene, label_rec)
        label_lookup = {lb.label_id: lb for lb in label_rec.labels}

        for item in scored:
            lb = label_lookup.get(item["label_id"])
            flagged_low = False
            if lb is not None:
                if lb.type in ("CAM_TRANSLATION", "CAM_ROTATION"):
                    flagged_low = "camera" in low_conf_entities
                else:
                    eids = [label_rec.entity_map.get(d, "") for d in (lb.subject, lb.a, lb.b) if d]
                    flagged_low = any(e in low_conf_entities for e in eids)
            rows.append(
                {
                    "clip_id": clip_id,
                    "source": label_rec.source,
                    "group": group,
                    "difficulty": "|".join(label_rec.difficulty) or "easy",
                    "system": system,
                    "label_id": item["label_id"],
                    "type": item["type"],
                    "pred": item.get("pred"),
                    "truth": item.get("truth"),
                    "correct": bool(item["correct"]),
                    "unresolved": bool(item.get("unresolved", False)),
                    "low_conf": flagged_low,
                }
            )

        if system == "full":
            log_path = run_dir / "loop_log.json"
            if log_path.exists():
                log_data = json.loads(log_path.read_text(encoding="utf-8"))
                accepted_rounds = sum(1 for r in log_data if r.get("round", 0) > 0 and r.get("accepted"))
                total_regs = 0
                for r in log_data:
                    if r.get("round", 0) <= 0:
                        continue
                    reg = r.get("regressions", 0)
                    total_regs += reg["count"] if isinstance(reg, dict) else int(reg)
                total_tokens = sum(int(r.get("tokens", 0)) for r in log_data)
                total_latency = sum(float(r.get("latency_s", 0.0)) for r in log_data)
                loop_rows.append(
                    {
                        "clip_id": clip_id,
                        "group": group,
                        "rounds_accepted": accepted_rounds,
                        "rounds_attempted": max(0, len(log_data) - 1),
                        "regressions": total_regs,
                        "tokens": total_tokens,
                        "latency_s": total_latency,
                    }
                )
            checks_r0 = run_dir / "checks_r0.json"
            if checks_r0.exists():
                for c in load_checks(checks_r0):
                    if c.control and c.result and c.result.p_yes is not None:
                        control_p_yes.append(c.result.p_yes)

df = pd.DataFrame(rows)
loop_df = pd.DataFrame(loop_rows)
print(f"Loaded {len(df)} scored label evaluations across {len(primary_by_clip)} {SPLIT} clips.")

## 3. Chart 1 — Baseline vs Loop Accuracy per Label Type (95% Clip-Bootstrap CIs)

In [ ]:
if not df.empty:
    types = sorted(df["type"].unique())
    x = np.arange(len(types))
    width = 0.36

    base_stats = [bootstrap_ci_by_clip(df[(df["system"] == "baseline") & (df["type"] == t)]) for t in types]
    full_stats = [bootstrap_ci_by_clip(df[(df["system"] == "full") & (df["type"] == t)]) for t in types]

    fig, ax = plt.subplots(figsize=(10, 4.8))
    b_means = [s[0] * 100 for s in base_stats]
    f_means = [s[0] * 100 for s in full_stats]
    b_err = [[(s[0] - s[1]) * 100 for s in base_stats], [(s[2] - s[0]) * 100 for s in base_stats]]
    f_err = [[(s[0] - s[1]) * 100 for s in full_stats], [(s[2] - s[0]) * 100 for s in full_stats]]

    ax.bar(
        x - width / 2, b_means, width, yerr=b_err, capsize=4, label="Baseline (Argon r0)", color="#90a4ae"
    )
    ax.bar(
        x + width / 2, f_means, width, yerr=f_err, capsize=4, label="God-Shot Loop (final)", color="#1e88e5"
    )
    ax.set_xticks(x)
    ax.set_xticklabels(types, rotation=15, ha="right")
    ax.set_ylabel("Accuracy vs Human Labels (%)")
    ax.set_ylim(0, 108)
    ax.set_title(f"Baseline vs Verify-and-Correct Loop by Label Type ({SPLIT} split, 95% clip bootstrap CI)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGS_DIR / "accuracy_by_type.png")
    plt.show()
else:
    print("No scored runs found yet — run eval/run_all.py first.")

## 4. Chart 2 — CameraBench (Motion Only) vs Own Clips (Layout + Motion) & Difficulty Breakdown

In [ ]:
if not df.empty:
    groups = [g for g in ("camerabench", "own_clips") if (df["group"] == g).any()]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

    xg = np.arange(len(groups))
    w = 0.35
    b_g = [bootstrap_ci_by_clip(df[(df["system"] == "baseline") & (df["group"] == g)]) for g in groups]
    f_g = [bootstrap_ci_by_clip(df[(df["system"] == "full") & (df["group"] == g)]) for g in groups]
    axes[0].bar(xg - w / 2, [s[0] * 100 for s in b_g], w, label="Baseline", color="#90a4ae")
    axes[0].bar(xg + w / 2, [s[0] * 100 for s in f_g], w, label="Loop", color="#1e88e5")
    axes[0].set_xticks(xg)
    group_labels = [
        "CameraBench\n(motion only)" if g == "camerabench" else "Own Clips\n(layout + motion)"
        for g in groups
    ]
    axes[0].set_xticklabels(group_labels)
    axes[0].set_ylim(0, 108)
    axes[0].set_ylabel("Accuracy (%)")
    axes[0].set_title("By Dataset Group")
    axes[0].legend()

    diffs = sorted(df["difficulty"].unique())
    xd = np.arange(len(diffs))
    b_d = [df[(df["system"] == "baseline") & (df["difficulty"] == d)]["correct"].mean() * 100 for d in diffs]
    f_d = [df[(df["system"] == "full") & (df["difficulty"] == d)]["correct"].mean() * 100 for d in diffs]
    axes[1].bar(xd - w / 2, b_d, w, label="Baseline", color="#90a4ae")
    axes[1].bar(xd + w / 2, f_d, w, label="Loop", color="#1e88e5")
    axes[1].set_xticks(xd)
    axes[1].set_xticklabels(diffs, rotation=20, ha="right")
    axes[1].set_ylim(0, 108)
    axes[1].set_title("By Difficulty Tag")

    fig.tight_layout()
    fig.savefig(FIGS_DIR / "accuracy_by_group.png")
    plt.show()

## 5. Chart 3 — Verifier Calibration (5-Bin Reliability Diagram, ECE, Brier) & Selective Accuracy

In [ ]:
vvh_path = RUNS_DIR / f"verifier_vs_human_{SPLIT}.json"
cal = {"ece": float("nan"), "brier": float("nan"), "bins": []}
vvh_acc = float("nan")

if vvh_path.exists():
    vvh = json.loads(vvh_path.read_text(encoding="utf-8"))
    vdf = pd.DataFrame(vvh)
    if not vdf.empty and "correct" in vdf:
        vvh_acc = float(vdf["correct"].mean())
    has_noul = (not vdf.empty) and ("p_yes" in vdf)
    noul_df = vdf.dropna(subset=["p_yes", "truth_bool"]) if has_noul else pd.DataFrame()
    if not noul_df.empty:
        probs_arr = noul_df["p_yes"].to_numpy(dtype=float)
        truth_arr = noul_df["truth_bool"].to_numpy(dtype=float)
        cal = calibration_stats(probs_arr, truth_arr)
        fig, ax = plt.subplots(figsize=(5.2, 4.8))
        ax.plot([0, 1], [0, 1], "--", color="#78909c", label="Perfect calibration")
        confs = [b["conf"] for b in cal["bins"]]
        accs = [b["acc"] for b in cal["bins"]]
        ax.plot(confs, accs, "o-", color="#1e88e5", linewidth=2, label="Decisions API (noul)")
        ax.set_xlabel("Mean predicted p(yes)")
        ax.set_ylabel("Empirical positive rate (human truth)")
        ax.set_title(f"Reliability Diagram (ECE={cal['ece']:.3f}, Brier={cal['brier']:.3f})")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.legend()
        fig.tight_layout()
        fig.savefig(FIGS_DIR / "calibration.png")
        plt.show()
else:
    print(f"Verifier-vs-human file not found yet: {vvh_path}")

if not df.empty:
    full_df = df[df["system"] == "full"]
    hi = full_df[~full_df["low_conf"]]
    lo = full_df[full_df["low_conf"]]
    if not hi.empty or not lo.empty:
        fig, ax = plt.subplots(figsize=(5.5, 4.2))
        hi_pct = hi["correct"].mean() * 100 if not hi.empty else 0.0
        lo_pct = lo["correct"].mean() * 100 if not lo.empty else 0.0
        labels_x = [f"High confidence\n(n={len(hi)})", f"Flagged '?' low_conf\n(n={len(lo)})"]
        ax.bar(labels_x, [hi_pct, lo_pct], color=["#43a047", "#ef5350"], width=0.5)
        ax.set_ylabel("Accuracy vs Human Labels (%)")
        ax.set_ylim(0, 108)
        ax.set_title("Selective Accuracy: Does It Know When It's Wrong?")
        fig.tight_layout()
        fig.savefig(FIGS_DIR / "selective_accuracy.png")
        plt.show()

## 6. Chart 4 — Loop Behavior (Rounds-to-Converge, Regressions, Share Hurt, Latency)

In [ ]:
if not loop_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.0))
    counts = Counter(loop_df["rounds_accepted"])
    xs = sorted(counts)
    axes[0].bar([str(k) for k in xs], [counts[k] for k in xs], color="#1e88e5")
    axes[0].set_xlabel("Accepted Correction Rounds")
    axes[0].set_ylabel("Clips")
    axes[0].set_title("Rounds to Converge")

    n_bins = min(10, max(3, len(loop_df)))
    axes[1].hist(loop_df["latency_s"], bins=n_bins, color="#5c6bc0", edgecolor="white")
    axes[1].set_xlabel("Loop Latency (s / clip)")
    axes[1].set_ylabel("Clips")
    axes[1].set_title(f"Latency (median {loop_df['latency_s'].median():.1f} s)")
    fig.tight_layout()
    fig.savefig(FIGS_DIR / "loop_behavior.png")
    plt.show()

## 7. Export `eval/results.md` and `eval/failures.md` (Slide Source of Truth)

In [ ]:
if not df.empty:
    base_df = df[df["system"] == "baseline"]
    full_df = df[df["system"] == "full"]
    b_all = bootstrap_ci_by_clip(base_df)
    f_all = bootstrap_ci_by_clip(full_df)
    sign = paired_sign_test(
        base_df.groupby("clip_id")["correct"].mean(),
        full_df.groupby("clip_id")["correct"].mean(),
    )
    pct_hurt = (sign["losses"] / sign["n_clips"] * 100.0) if sign["n_clips"] else 0.0
    hi = full_df[~full_df["low_conf"]]
    lo = full_df[full_df["low_conf"]]
    hi_acc = hi["correct"].mean() if not hi.empty else float("nan")
    lo_acc = lo["correct"].mean() if not lo.empty else float("nan")
    yes_bias = float(np.mean(control_p_yes)) if control_p_yes else float("nan")
    n_cb = df[df["group"] == "camerabench"]["clip_id"].nunique()
    n_own = df[df["group"] == "own_clips"]["clip_id"].nunique()

    sign_line = (
        f"- **Paired sign test (by clip):** {sign['wins']} improved / {sign['losses']} hurt / "
        f"{sign['ties']} tied (p = {sign['p_value']:.4f}; loop hurt {pct_hurt:.1f}% of clips)"
    )
    lines = [
        f"# God-Shot Evaluation Results (`{SPLIT}` split)",
        "",
        f"- **Generated:** {datetime.now().isoformat(timespec='seconds')}",
        f"- **Prompt/threshold freeze time:** {FREEZE_TIME}",
        f"- **Clips scored:** {sign['n_clips']} ({n_cb} CameraBench, {n_own} own clips)",
        "",
        "## Headline: Baseline vs Verify-and-Correct Loop (vs Human Labels)",
        f"- **Baseline (Argon r0):** {b_all[0] * 100:.1f}% "
        f"(95% CI [{b_all[1] * 100:.1f}%, {b_all[2] * 100:.1f}%])",
        f"- **Full Loop (final):** {f_all[0] * 100:.1f}% "
        f"(95% CI [{f_all[1] * 100:.1f}%, {f_all[2] * 100:.1f}%])",
        sign_line,
        "",
        "## Per-Label-Type Accuracy",
        "| Label Type | Baseline (95% CI) | Full Loop (95% CI) | N |",
        "|---|---|---|---|",
    ]
    for t in sorted(df["type"].unique()):
        bs = bootstrap_ci_by_clip(base_df[base_df["type"] == t])
        fs = bootstrap_ci_by_clip(full_df[full_df["type"] == t])
        n_t = int((full_df["type"] == t).sum())
        b_str = f"{bs[0] * 100:.1f}% [{bs[1] * 100:.1f}, {bs[2] * 100:.1f}]"
        f_str = f"{fs[0] * 100:.1f}% [{fs[1] * 100:.1f}, {fs[2] * 100:.1f}]"
        lines.append(f"| `{t}` | {b_str} | {f_str} | {n_t} |")

    vvh_line = (
        f"- **Verifier vs Human accuracy:** {vvh_acc * 100:.1f}%"
        if not math.isnan(vvh_acc)
        else "- **Verifier vs Human accuracy:** pending `verifier_vs_human_*.json`"
    )
    cal_line = (
        f"- **Calibration (noul):** ECE = {cal['ece']:.3f}, Brier = {cal['brier']:.3f}"
        if not math.isnan(cal["ece"])
        else "- **Calibration:** pending"
    )
    bias_line = (
        f"- **Mirrored-control yes-bias (mean p_yes):** {yes_bias:.3f} (n={len(control_p_yes)})"
        if control_p_yes
        else "- **Mirrored-control yes-bias:** pending"
    )
    sel_line = (
        f"- **Selective accuracy:** high-confidence = {hi_acc * 100:.1f}% (n={len(hi)}) "
        f"vs flagged low_confidence = {lo_acc * 100:.1f}% (n={len(lo)})"
    )
    lines.extend(
        ["", "## Verifier, Calibration & Selective Accuracy", vvh_line, cal_line, bias_line, sel_line]
    )
    if not loop_df.empty:
        med_lat = loop_df["latency_s"].median()
        mean_lat = loop_df["latency_s"].mean()
        lines.extend(
            [
                "",
                "## Loop Behavior & Efficiency",
                f"- **Median latency:** {med_lat:.1f} s/clip (mean {mean_lat:.1f} s)",
                f"- **Mean accepted rounds:** {loop_df['rounds_accepted'].mean():.2f}",
                f"- **Total regressions across attempted rounds:** {int(loop_df['regressions'].sum())}",
                f"- **Mean tokens / clip:** {loop_df['tokens'].mean():.0f}",
            ]
        )

    RESULTS_MD.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"Wrote {RESULTS_MD}")

    failed_rows = full_df[~full_df["correct"]]
    flines = [
        f"# Failed Labels (`{SPLIT}` split — pick 4 for `assets/img/failure_gallery.png`)",
        "",
        "| Clip | Difficulty | Label ID | Type | Scene Claim | Human Truth | Low Conf? | Thumbnail |",
        "|---|---|---|---|---|---|---|---|",
    ]
    for _, r in failed_rows.iterrows():
        thumb = f"frames/{r['clip_id']}/001000.jpg"
        row_md = (
            f"| `{r['clip_id']}` | {r['difficulty']} | {r['label_id']} | `{r['type']}` "
            f"| `{r['pred']}` | `{r['truth']}` | {r['low_conf']} | `{thumb}` |"
        )
        flines.append(row_md)
    FAILURES_MD.write_text("\n".join(flines) + "\n", encoding="utf-8")
    print(f"Wrote {FAILURES_MD} ({len(failed_rows)} failed labels)")